In [22]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [29]:
from pathlib import Path
import numpy as np
import pandas as pd

YEARS = [str(y) for y in range(2019, 2026)]

def read_prices(path: str | Path, typ: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8-sig")
    data = df.melt(id_vars="Produkt", value_vars=YEARS, var_name="rok", value_name="cena")
    data["rok"] = data["rok"].astype(int)
    data["cena"] = pd.to_numeric(data["cena"], errors="coerce")
    data["typ"] = typ
    return data

def create_features(food_path: str | Path, fuel_path: str | Path) -> pd.DataFrame:
    df = pd.concat([read_prices(food_path, "żywność"), read_prices(fuel_path, "paliwo")], ignore_index=True)
    df = df.sort_values(["typ", "Produkt", "rok"]).reset_index(drop=True)
    g = df.groupby(["typ", "Produkt"], sort=False)["cena"]
    #cena z ubiegłego roku, niezależna od ceny z roku bieżącego.
    df["cena_lag1"] = g.shift(1)
    #zmiana procentowa r/r; informacja opisowa dla t oraz predyktor ceny t+1.
    df["zmiana_roczna_proc"] = (df["cena"] / df["cena_lag1"] - 1) * 100
    #indeks ceny względem roku bazowego 2019.
    base = df[df["rok"].eq(2019)][["typ", "Produkt", "cena"]].rename(columns={"cena":"cena_2019"})
    df = df.merge(base, on=["typ", "Produkt"], how="left", validate="many_to_one")
    df["indeks_2019_100"] = df["cena"] / df["cena_2019"] * 100
    #3-letnia średnia z cen wcześniejszych lat (nie obejmuje roku t).
    df["srednia_poprzednich_3_lat"] = df.groupby(["typ", "Produkt"], sort=False)["cena"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    #różnica dynamiki produktu względem średniej dynamiki 3 paliw.
    fuel_changes = (df[df["typ"].eq("paliwo")].groupby("rok")["zmiana_roczna_proc"]
                    .mean().rename("srednia_zmiana_paliw_proc"))
    df = df.join(fuel_changes, on="rok")
    df["roznica_dynamiki_vs_paliwa_pp"] = df["zmiana_roczna_proc"] - df["srednia_zmiana_paliw_proc"]
    #skownik opisuje tylko pięć nowych cech; kolumny pomocnicze usuwamy.
    df = df.drop(columns=["cena_2019", "srednia_zmiana_paliw_proc"])
    return df

from pathlib import Path

base = Path.cwd()

result = create_features(
    base / "jedzenie_clean.csv",
    base / "paliwa_jak_jedzenie.csv"
)

result.to_csv(
    base / "cechy_kluczowe.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    index=False
)

print(result.head(10).to_string(index=False))

    Produkt  rok    cena    typ  cena_lag1  zmiana_roczna_proc  indeks_2019_100  srednia_poprzednich_3_lat  roznica_dynamiki_vs_paliwa_pp
ONEkodiesel 2019 4.02625 paliwo        NaN                 NaN       100.000000                        NaN                            NaN
ONEkodiesel 2020 3.34900 paliwo    4.02625          -16.820863        83.179137                        NaN                      -2.923774
ONEkodiesel 2021 4.22825 paliwo    3.34900           26.254106       105.017075                   3.687625                      -0.823410
ONEkodiesel 2022 6.69225 paliwo    4.22825           58.274700       166.215461                   3.867833                      13.836137
ONEkodiesel 2023 5.12700 paliwo    6.69225          -23.388995       127.339336                   4.756500                      -6.335957
ONEkodiesel 2024 4.98450 paliwo    5.12700           -2.779403       123.800062                   5.349167                       0.011045
ONEkodiesel 2025 4.66350 paliwo   